# 02 — FinBERT Baseline & Leakage-Safe Evaluation

This notebook is the first **model experiment** for RiskPulse.

Goals:
1. Load the reconstructed historical tweet dataset.
2. Run **ProsusAI/FinBERT** on a reproducible sample/full dataset.
3. Produce the required continuous sentiment score:
   `positive_probability - negative_probability`.
4. Compare FinBERT descriptively against the dataset's existing LSTM/TextBlob polarity signals.
5. Examine the relationship between sentiment and **future** returns strictly as an offline evaluation.

**Important:** `LSTM_POLARITY` and `TEXTBLOB_POLARITY` are reference/baseline signals, not ground-truth labels. Future-return columns are never used as live inference features.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from ingestion.historical_tweets import load_csv
from nlp.sentiment import FinBERTSentiment
from evaluation.sentiment_evaluation import add_finbert_score, correlation_table, bucketed_return_summary

RAW = ROOT / "data" / "raw"
CSV_FULL = ROOT / "data" / "processed" / "full_dataset-release.csv"
CSV_REDUCED = ROOT / "data" / "processed" / "reduced_dataset-release.csv"

print("Project root:", ROOT)
print("Full CSV:", CSV_FULL)
print("Reduced CSV:", CSV_REDUCED)

## 1. Dataset policy

For the final system, **we target the full dataset** because it gives roughly 862k historical observations and 101 stocks.

We will **not load all 862k tweets into the transformer in one batch**. Instead:

- use the full dataset for offline evaluation/training experiments;
- process it in batches/chunks;
- cache FinBERT outputs;
- use a deterministic sample for rapid notebook iteration;
- use the reduced dataset for quick smoke tests.

This gives us both robustness and practical development speed.

In [ ]:
# If the CSVs have not yet been extracted, run the audit notebook first.
# For this notebook, prefer the reconstructed CSV produced by the audit pipeline.

if CSV_FULL.exists():
    df = load_csv(CSV_FULL)
elif CSV_REDUCED.exists():
    df = load_csv(CSV_REDUCED)
else:
    raise FileNotFoundError(
        "Expected reconstructed CSV under data/processed/. "
        "Run notebooks/01_tweet_dataset_audit.ipynb first."
    )

print(df.shape)
display(df.head(3))

## 2. Rapid development sample

The first model run should use a fixed seed so results are reproducible.

Later, we will run the same pipeline over the complete full dataset in chunks and persist predictions.

In [ ]:
SAMPLE_SIZE = min(5000, len(df))
sample = df.sample(SAMPLE_SIZE, random_state=42).copy()

print("Sample:", sample.shape)
print(sample["STOCK"].nunique(), "unique stocks")
print(sample["DATE"].min(), "to", sample["DATE"].max())

## 3. Load FinBERT

First run downloads the Hugging Face model. Subsequent runs should use the local Hugging Face cache.

In [ ]:
sentiment_model = FinBERTSentiment()
print("Device:", sentiment_model.device)

In [ ]:
scored = add_finbert_score(
    sample,
    sentiment_model,
    text_col="TWEET",
    batch_size=16,
)

display(
    scored[
        [
            "TWEET", "STOCK",
            "FINBERT_LABEL",
            "FINBERT_POSITIVE",
            "FINBERT_NEGATIVE",
            "FINBERT_NEUTRAL",
            "FINBERT_SCORE",
            "LSTM_POLARITY",
            "TEXTBLOB_POLARITY",
            "1_DAY_RETURN",
            "7_DAY_RETURN",
        ]
    ].head(20)
)

## 4. Required Risk Engine sentiment representation

FinBERT probabilities are converted into the continuous range `[-1, +1]`:

`sentiment_score = P(positive) - P(negative)`

This is the signal consumed by the downstream Risk Engine.

In [ ]:
scored["FINBERT_SCORE"].describe()

## 5. Compare against existing baselines

This is a **descriptive comparison**, not supervised validation, because the dataset's LSTM/TextBlob fields are themselves model-generated polarity signals.

In [ ]:
display(correlation_table(scored))

## 6. Sentiment buckets vs future returns

This section is deliberately retrospective.

The model sees only `TWEET` when generating `FINBERT_SCORE`. We then compare that score with `1_DAY_RETURN` and `7_DAY_RETURN` that occurred after the tweet.

Do not use these return columns as input features to the live model.

In [ ]:
display(
    bucketed_return_summary(
        scored,
        score_col="FINBERT_SCORE",
        return_col="1_DAY_RETURN",
        bins=5,
    )
)

display(
    bucketed_return_summary(
        scored,
        score_col="FINBERT_SCORE",
        return_col="7_DAY_RETURN",
        bins=5,
    )
)

## 7. Next experiment

This baseline is intentionally not the final Risk Engine.

Next we will add:
- finance-aware entity resolution;
- event classification;
- event clustering/deduplication;
- source credibility;
- abnormal-attention features;
- market context;
- impact score calibration;
- confidence;
- and ultimately the portfolio stress engine.

The historical return fields remain evaluation-only.